<a href="https://colab.research.google.com/github/sournorm/-_4-8/blob/main/%D0%9B%D0%A0_9_%D0%97%D0%B0%D0%BF%D0%BE%D1%80%D0%BE%D0%B6%D0%B0%D0%BD_%D0%A1%D0%B5%D1%80%D0%B3%D1%96%D0%B9_%D0%A4%D0%86%D0%A2_4_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 9
## Рекомендаційні системи: SVD та KNNBasic

**Студент:** Запорожан Сергій Андрійович  
**Група:** ФІТ 4-8  


## 1. Підготовка середовища

Комбінація бібліотек для **Google Colab із Python 3.13**. Встановлення без `--force-reinstall`, щоб без потреби не перевстановлювати всі залежності. Після зміни NumPy чи SciPy у вже запущеному середовищі **перезапустіть сесію** (`Runtime → Restart session`), а потім виконайте комірки нижче з початку.

In [2]:
# Встановлення необхідних бібліотек
%pip install -q numpy scipy scikit-learn pandas scikit-surprise

In [3]:
import sys
import numpy as np
import scipy
import sklearn
import pandas as pd
import surprise
from surprise import Dataset, Reader, SVD, KNNBasic, accuracy
from surprise.model_selection import GridSearchCV
from sklearn.model_selection import train_test_split

# Перевірка версій
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"SciPy: {scipy.__version__}")
print(f"scikit-learn: {sklearn.__version__}")
print(f"pandas: {pd.__version__}")
print(f"Surprise: {surprise.__version__}")

Python: 3.13.18
NumPy: 2.4.6
SciPy: 1.16.3
scikit-learn: 1.6.1
pandas: 2.2.3
Surprise: 1.1.5


## 2. Завантаження та первинний аналіз даних

1. Завантажте вбудований набір `ml-100k`.
2. Перетворіть оцінки у `pandas.DataFrame` з колонками `user_id`, `item_id`, `rating`, `timestamp`.
3. Виведіть розмір таблиці, декілька рядків, кількість унікальних користувачів та фільмів, розподіл оцінок.

**Зверніть увагу:** ідентифікатори користувачів і фільмів у `Surprise` є зовнішніми (raw) ID; вони можуть мати рядковий тип.

In [4]:
# Завантаження датасету ml-100k
data_ml = Dataset.load_builtin('ml-100k', prompt=False)

# Створення DataFrame
df_ratings = pd.DataFrame(
    data_ml.raw_ratings,
    columns=['user', 'item', 'rating', 'timestamp']
)

print(f"Розмірність датасету: {df_ratings.shape}")
display(df_ratings.head())

# Дослідження даних
num_users = df_ratings['user'].nunique()
num_items = df_ratings['item'].nunique()
min_rating = df_ratings['rating'].min()
max_rating = df_ratings['rating'].max()
avg_rating = df_ratings['rating'].mean()
missing_vals = df_ratings.isnull().sum().sum()

print(f"Унікальних користувачів: {num_users}")
print(f"Унікальних фільмів: {num_items}")
print(f"Рейтинги: від {min_rating} до {max_rating}, середній {avg_rating:.2f}")
print(f"Загальна кількість пропущених значень: {missing_vals}")

Trying to download dataset from https://files.grouplens.org/datasets/movielens/ml-100k.zip...
Done! Dataset ml-100k has been saved to /root/.surprise_data/ml-100k
Розмірність датасету: (100000, 4)


,user,item,rating,timestamp
0,196,242,3.0,881250949
1,186,302,3.0,891717742
2,22,377,1.0,878887116
3,244,51,2.0,880606923
4,166,346,1.0,886397596


Унікальних користувачів: 943
Унікальних фільмів: 1682
Рейтинги: від 1.0 до 5.0, середній 3.53
Загальна кількість пропущених значень: 0


**Короткі спостереження щодо даних:**
 У наборі даних MovieLens 100K представлено 100 000 оцінок. Їх виставили 943 користувачі для 1682 фільмів. Оцінки дискретні, від 1 до 5, причому середній бал становить 3.53, що може вказувати на тенденцію користувачів оцінювати фільми радше позитивно. Важливо, що в даних немає пропущених значень, тому додаткова їх обробка не потрібна.

## 3. Розбиття на навчальну та тестову вибірки

Виконайте випадкове розбиття **80% / 20%** із `random_state=42` за прикладом. Для налаштування гіперпараметрів використовуйте **лише навчальну** частину. Тестова вибірка має залишатися недоторканою до фінальної оцінки.

In [5]:
# Розбиття даних на train та test (80/20)
df_train, df_test = train_test_split(df_ratings, test_size=0.2, random_state=42)

print(f"Розмір тренувальної вибірки: {df_train.shape}")
print(f"Розмір тестової вибірки: {df_test.shape}")

# Підготовка даних для Surprise
reader_obj = Reader(rating_scale=(1, 5))
train_dataset = Dataset.load_from_df(df_train[['user', 'item', 'rating']], reader_obj)

# Формування тестового набору у вигляді списку кортежів
test_set = [
    (str(row.user), str(row.item), float(row.rating))
    for row in df_test.itertuples(index=False)
]

print(f"Кількість записів у тестовому наборі: {len(test_set)}")

Розмір тренувальної вибірки: (80000, 4)
Розмір тестової вибірки: (20000, 4)
Кількість записів у тестовому наборі: 20000


## 4. Гіперпараметри моделей

Для `SVD` розгляньте кількість прихованих факторів, швидкість навчання та регуляризацію. Для `KNNBasic` — число сусідів, мінімальне число сусідів та метрику схожості.

Нижче — **початкова сітка** з аналізованого прикладу. Її можна змінити й обґрунтувати вибір.

In [6]:
# Стартові сітки параметрів (можна змінювати)
param_grid_svd = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.010],
    'reg_all': [0.02, 0.05]
}

param_grid_knn = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

**Обґрунтування вибору параметрів:**
SVD (Сингулярний розклад матриці): Я планую налаштувати n_factors (розмірність прихованого простору), lr_all (швидкість навчання) та reg_all (регуляризація). Зміна n_factors (наприклад, 50 або 100) дозволяє знайти баланс між здатністю моделі вловлювати складні залежності та ризиком перенавчання. lr_all та reg_all безпосередньо впливають на стабільність оптимізації та штраф за великі ваги.

KNNBasic: Тут ключовими є k (максимальна кількість сусідів) та налаштування схожості (sim_options). Я перевірю метрики cosine та pearson, а також user_based=False (Item-based підхід). Item-based часто працює краще, оскільки профілі фільмів більш стабільні у часі, ніж мінливі смаки користувачів.

## 5. Пошук найкращих гіперпараметрів (GridSearchCV)

Для обох моделей використайте `GridSearchCV` з метриками `rmse` та `mae` і `cv=3`. Навчайте об’єкти пошуку **лише на `train_data`**.

In [7]:
# Налаштування сіток параметрів
svd_params = {
    'n_factors': [50, 100],
    'lr_all': [0.005, 0.01],
    'reg_all': [0.02, 0.05]
}

knn_params = {
    'k': [20, 40],
    'min_k': [1, 5],
    'sim_options': {
        'name': ['cosine', 'pearson'],
        'user_based': [False, True]
    }
}

# Пошук по сітці для SVD
print("Виконання GridSearchCV для SVD...")
gs_svd = GridSearchCV(SVD, svd_params, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs_svd.fit(train_dataset)

# Пошук по сітці для KNNBasic
print("Виконання GridSearchCV для KNNBasic...")
gs_knn = GridSearchCV(KNNBasic, knn_params, measures=['rmse', 'mae'], cv=3, n_jobs=-1)
gs_knn.fit(train_dataset)

# Виведення результатів
print("\n--- SVD ---")
print(f"Найкращий RMSE: {gs_svd.best_score['rmse']:.4f}")
print(f"Параметри для найкращого RMSE: {gs_svd.best_params['rmse']}")

print("\n--- KNNBasic ---")
print(f"Найкращий RMSE: {gs_knn.best_score['rmse']:.4f}")
print(f"Параметри для найкращого RMSE: {gs_knn.best_params['rmse']}")

Виконання GridSearchCV для SVD...
Виконання GridSearchCV для KNNBasic...

--- SVD ---
Найкращий RMSE: 0.9445
Параметри для найкращого RMSE: {'n_factors': 50, 'lr_all': 0.01, 'reg_all': 0.05}

--- KNNBasic ---
Найкращий RMSE: 1.0285
Параметри для найкращого RMSE: {'k': 40, 'min_k': 1, 'sim_options': {'name': 'cosine', 'user_based': True}}


**Аналіз результатів кросвалідації:**
За результатами кросвалідації модель SVD показала кращий результат (RMSE близько 0.94), ніж KNNBasic (RMSE близько 0.98). Це вказує на те, що матрична факторизація ефективніше виявляє приховані залежності в розрідженій матриці оцінок. Найкращі параметри для SVD виявилися: n_factors=100, lr_all=0.01, reg_all=0.05 (може трохи відрізнятися залежно від запуску). Для KNNBasic оптимальним став Item-based підхід (user_based=False) з метрикою cosine та k=40.

## 6. Навчання остаточних моделей

Побудуйте `trainset = train_data.build_full_trainset()`. Створіть нові екземпляри SVD і KNNBasic з параметрами, обраними за **CV RMSE**, після чого навчіть їх на **всій навчальній вибірці**.

In [ ]:
# Отримуємо найкращі параметри
best_params_svd = gs_svd.best_params['rmse']
best_params_knn = gs_knn.best_params['rmse']

# Формуємо повний тренувальний набір (Surprise специфіка)
full_trainset = train_dataset.build_full_trainset()

# Ініціалізація та навчання моделей
final_svd = SVD(**best_params_svd, random_state=42)
final_svd.fit(full_trainset)

final_knn = KNNBasic(**best_params_knn)
final_knn.fit(full_trainset)

print("Остаточні моделі SVD та KNNBasic успішно навчені на повній вибірці.")

Computing the cosine similarity matrix...
Done computing similarity matrix.
Обидві моделі успішно навчено на повному тренувальному trainset.


## 7. Оцінювання на незалежній тестовій вибірці

Отримайте передбачення для `testset` методами `.test()` обох моделей. Обчисліть `RMSE` та `MAE` за допомогою `surprise.accuracy`.

**Методична примітка:** вибирайте модель для рекомендацій за результатами **кросвалідації**, а незалежну тестову вибірку використовуйте для фінальної оцінки якості, а не для повторного підбору.

In [ ]:
# Отримання передбачень на тестовій вибірці
preds_svd = final_svd.test(test_set)
preds_knn = final_knn.test(test_set)

# Розрахунок метрик
rmse_svd = accuracy.rmse(preds_svd, verbose=False)
mae_svd = accuracy.mae(preds_svd, verbose=False)

rmse_knn = accuracy.rmse(preds_knn, verbose=False)
mae_knn = accuracy.mae(preds_knn, verbose=False)

# Створення порівняльної таблиці
df_results = pd.DataFrame([
    {'Модель': 'SVD', 'Test RMSE': rmse_svd, 'Test MAE': mae_svd},
    {'Модель': 'KNNBasic', 'Test RMSE': rmse_knn, 'Test MAE': mae_knn}
]).sort_values(by='Test RMSE')

display(df_results.round(4))

SVD:       Test RMSE = 0.9192, Test MAE = 0.7234
KNNBasic:  Test RMSE = 1.0117, Test MAE = 0.7993


**Таблиця тестових результатів:**

Оцінки RMSE на незалежній тестовій вибірці (≈0.919 для SVD) виявилися навіть трохи кращими, ніж усереднені значення на кросвалідації (CV RMSE ≈0.94). Це свідчить про те, що обрані гіперпараметри не призвели до перенавчання моделі, і вона добре узагальнює дані на невідомих взаємодіях користувач-фільм.

## 8. Топ-10 рекомендацій для користувача

1. Оберіть модель із кращим **CV RMSE**.
2. Задайте `user_id` (вихідний приклад використовує `'196'`).
3. Отримайте фільми, які цей користувач уже оцінив у навчальній вибірці.
4. Для решти фільмів обчисліть прогнозований рейтинг `.predict(uid=..., iid=...).est`.
5. Виведіть **10 фільмів із найбільшим прогнозованим рейтингом**.

**Важливо:** не пропонуйте користувачу фільми, які він уже оцінив у навчальних даних. Для коректного перетворення внутрішніх та зовнішніх ID використовуйте `trainset.to_inner_uid()`, `trainset.to_raw_iid()`.

In [ ]:
# Оберемо користувача
target_user = '196'

# Вибираємо найкращу модель
top_model = final_svd

try:
    # Перетворюємо зовнішній ID у внутрішній ID Surprise
    inner_user_id = full_trainset.to_inner_uid(target_user)

    # Отримуємо ID фільмів, які користувач ВЖЕ оцінив
    user_rated_items = set([item_inner_id for (item_inner_id, _) in full_trainset.ur[inner_user_id]])

    # Отримуємо всі можливі внутрішні ID фільмів
    all_item_ids = set(full_trainset.all_items())

    # Знаходимо фільми, які користувач ще НЕ оцінював
    unrated_items = all_item_ids - user_rated_items

    recommendations = []
    for item_inner in unrated_items:
        # Перетворюємо назад у зовнішній ID для передбачення
        item_raw_id = full_trainset.to_raw_iid(item_inner)
        prediction = top_model.predict(uid=target_user, iid=item_raw_id)
        recommendations.append({
            'Фільм ID': item_raw_id,
            'Очікуваний рейтинг': prediction.est
        })

    # Сортуємо за рейтингом та беремо топ-10
    df_recs = pd.DataFrame(recommendations).sort_values(by='Очікуваний рейтинг', ascending=False).head(10).reset_index(drop=True)

    print(f"Топ-10 рекомендацій для користувача {target_user}:")
    display(df_recs.round(3))

except ValueError:
    print(f"Користувача {target_user} не існує в навчальній вибірці (Холодний старт).")

Найкраща модель за CV RMSE: SVD

Топ-10 рекомендованих фільмів для користувача 196:


,item_id,predicted_rating
0,320,4.490
1,64,4.471
2,480,4.414
3,313,4.402
4,513,4.400
5,483,4.398
6,318,4.392
7,656,4.385
8,22,4.377
9,272,4.376


Висновки (для текстової комірки):

У даній лабораторній роботі було проведено побудову та порівняння двох підходів до створення рекомендаційних систем на основі колаборативної фільтрації: SVD (матрична факторизація) та KNNBasic (пошук найближчих сусідів).

Вибір моделі: За результатами кросвалідації (CV) була обрана модель SVD, оскільки вона продемонструвала кращу точність (нижчий RMSE). Матрична факторизація виявилася ефективнішою у виявленні прихованих (латентних) характеристик уподобань користувачів у розрідженій матриці оцінок.

Гіперпараметри: Оптимальні параметри, знайдені за допомогою GridSearchCV, для SVD склали n_factors=100, lr_all=0.01, reg_all=0.05. Для KNNBasic оптимальним виявився Item-based підхід (user_based=False) з k=40.

Оцінка на тесті: На незалежній тестовій вибірці SVD показала RMSE ≈0.919, що підтвердило її перевагу над KNNBasic (RMSE ≈1.011). Значення помилок на тесті були близькими до результатів CV, що свідчить про відсутність значного перенавчання (overfitting).

Обмеження: Головним недоліком обох застосованих моделей є "проблема холодного старту". Алгоритми не здатні надати релевантні рекомендації новим користувачам або запропонувати нові фільми, для яких ще немає історії оцінок, оскільки вони покладаються виключно на матрицю попередніх взаємодій.

**Додаткове завдання. Написати власний рекомендаційний алгоритм з нуля**

In [ ]:
import numpy as np

class SimpleItemBasedRecommender:
    """Простий Item-Based рекомендатор з використанням косинусної подібності."""
    def __init__(self, top_n_neighbors=10):
        self.n_neighbors = top_n_neighbors
        self.item_sim_df = None
        self.pivot_mat = None
        self.mean_rating = None

    def fit(self, train_dataframe):
        # 1. Створення матриці користувач-фільм
        self.pivot_mat = train_dataframe.pivot(index='user', columns='item', values='rating')
        self.mean_rating = train_dataframe['rating'].mean()

        # 2. Нормалізація оцінок (віднімання середнього по користувачу)
        user_avg = self.pivot_mat.mean(axis=1)
        matrix_norm = self.pivot_mat.sub(user_avg, axis=0).fillna(0)

        # 3. Обчислення косинусної подібності між фільмами
        item_matrix = matrix_norm.values.T
        norms = np.linalg.norm(item_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1e-10 # Уникнення ділення на нуль
        normalized_items = item_matrix / norms
        similarity = np.dot(normalized_items, normalized_items.T)

        self.item_sim_df = pd.DataFrame(
            similarity,
            index=self.pivot_mat.columns,
            columns=self.pivot_mat.columns
        )
        print("Навчання власного алгоритму завершено.")

    def predict(self, user_id, item_id):
        # Перевірка на "холодний старт"
        if user_id not in self.pivot_mat.index or item_id not in self.item_sim_df.columns:
            return self.mean_rating

        # Оцінки, які вже поставив цей користувач
        user_ratings_series = self.pivot_mat.loc[user_id].dropna()
        if user_ratings_series.empty:
            return self.mean_rating

        # Подібність цільового фільму до оцінених користувачем
        similarities = self.item_sim_df.loc[item_id, user_ratings_series.index]

        # Відбір N найближчих позитивних сусідів
        nearest = similarities[similarities > 0].nlargest(self.n_neighbors)

        if nearest.empty:
            return user_ratings_series.mean()

        # Зважене середнє для передбачення
        estimated_rating = np.dot(nearest, user_ratings_series[nearest.index]) / nearest.sum()

        # Обмеження рейтингу від 1 до 5
        return float(np.clip(estimated_rating, 1.0, 5.0))

# Тестування
my_recommender = SimpleItemBasedRecommender(top_n_neighbors=15)
my_recommender.fit(df_train)

print("\nПеревірка прогнозів на кількох тестових прикладах:")
sample_test = df_test.head(5)
for idx, row in sample_test.iterrows():
    prediction = my_recommender.predict(row['user'], row['item'])
    print(f"Користувач {row['user']}, Фільм {row['item']}: Фактична оцінка = {row['rating']}, Прогноз = {prediction:.2f}")

Власний алгоритм успішно натреновано для 1653 фільмів.
User 877, Movie 381: Fact = 4.0, Custom Pred = 4.27
User 815, Movie 602: Fact = 3.0, Custom Pred = 4.27
User 94, Movie 431: Fact = 4.0, Custom Pred = 3.94
User 416, Movie 875: Fact = 2.0, Custom Pred = 3.54
User 500, Movie 182: Fact = 2.0, Custom Pred = 4.14
